### Getting Started With Langchain

In [5]:
import langchain

In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### 1: Simple LLM Call With Streaming

In [8]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage

In [9]:
model = init_chat_model("groq:llama-3.1-8b-instant")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001ACF02E8C20>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001ACF02E9940>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [10]:
# Old way to initialize the model, which is now deprecated in favor of the new method above.
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001ACF03B6210>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001ACF03B6C10>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [11]:
## Create messages
messages = [
    SystemMessage("You are a helpful AI assistant."),
    HumanMessage("What is the capital of France?")
]

In [12]:
## invoke the model
response = model.invoke(messages)
response

AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 49, 'total_tokens': 57, 'completion_time': 0.005769357, 'completion_tokens_details': None, 'prompt_time': 0.002624152, 'prompt_tokens_details': None, 'queue_time': 0.045110828, 'total_time': 0.008393509}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_ff2b098aaf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c7f89-11cb-7830-9885-b3a5ba8d5462-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 49, 'output_tokens': 8, 'total_tokens': 57})

In [13]:
print(response.content)

The capital of France is Paris.


In [14]:
model.invoke([HumanMessage("What is the capital of Germany?")])

AIMessage(content='The capital of Germany is Berlin.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 42, 'total_tokens': 50, 'completion_time': 0.006121661, 'completion_tokens_details': None, 'prompt_time': 0.001905323, 'prompt_tokens_details': None, 'queue_time': 0.045668617, 'total_time': 0.008026984}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c7f89-1b3e-7770-8b75-3a2091ad2e37-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 8, 'total_tokens': 50})

In [15]:
## Streaming example (Another way to invoke the model)
for chunk in model.stream(messages):
    print(chunk.content, end="", flush=True)

The capital of France is Paris.

### 2 : Dynamic Prompt Templates 

In [16]:
from langchain_core.prompts import ChatPromptTemplate

## Create translation app
translation_template = ChatPromptTemplate.from_messages({
    ("system", "You are a professional translator. Translate the following text {text} from {source_language} to {target_language}. Maintain the tone and style."),
    ("user", "{text}")
})

## using the template
prompt = translation_template.invoke({
    "source_language": "English",
    "target_language": "Spanish",
    "text": "Langchain makes building AI application incredibly easy!"
})

In [17]:
translated_response = model.invoke(prompt)
print(translated_response.content) 

Langchain hace que construir aplicaciones de inteligencia artificial sea increíblemente fácil!


### 3: Building Your First Chain

In [18]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

def create_script_chain():
    ## template for script generation
    script_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a creative scriptwriter. Write a short and enaging script based on a given theme, character and settings."),
        ("user", "Theme: {theme}\n Main Character: {character} \n Settings: {setting}")
    ])

    # template for story analysis
    analysis_prompt = ChatPromptTemplate.from_messages([
        ("system", "You're a literary critic. Analyze the script and provide insights."),
        ("user", "{script}")
    ])

    # Generates the script
    script_chain = (
        script_prompt
        | model
        | StrOutputParser()
    )

    # func to pass the script to analysis
    def analyze_script(script_text):
        return {"script": script_text}

    analysis_chain = (
        # script_chain
        # | RunnableLambda(analyze_script) # it will takes the script_chain and gives clean output 
        {'script': script_chain} # it only passes the chain (key:value pairs output)
        | analysis_prompt
        | model
        | StrOutputParser()
    )
    
    return analysis_chain

In [19]:
chain = create_script_chain()
chain

{
  script: ChatPromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a creative scriptwriter. Write a short and enaging script based on a given theme, character and settings.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, template='Theme: {theme}\n Main Character: {character} \n Settings: {setting}'), additional_kwargs={})])
          | ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001ACF02E8C20>, async_clie

In [26]:
# With RunnableLambda
chain = create_script_chain()
chain

ChatPromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a creative scriptwriter. Write a short and enaging script based on a given theme, character and settings.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, template='Theme: {theme}\n Main Character: {character} \n Settings: {setting}'), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002342CE946E0>, async_client=<groq.resources.cha

In [21]:
result = chain.invoke({
    "theme": "Ambition vs Reality",
    "character": "A middle-class B.Tech student who dreams of building a startup but struggles with academics, family expectations, and self-doubt.",
    "setting": "Tier-2 engineering college, Hostel room (late night coding scenes),Campus canteen (idea discussions),Placement season atmosphere, Tech fest / hackathon",
})
print("Script and Analysis :")
print(result)

Script and Analysis :
**Overall Impression**

The script presents a relatable and engaging narrative that explores the complexities of ambition, self-doubt, and the importance of friendship. The writer effectively conveys the struggles of a young protagonist navigating the pressures of academic and professional expectations. The script's pacing is well-balanced, moving seamlessly from moments of frustration to those of triumph.

**Character Development**

Rohan's character is well-crafted, showcasing a nuanced and realistic portrayal of a young adult struggling with self-doubt and fear. His transformation from a hesitant and uncertain individual to a determined and confident entrepreneur is authentic and compelling. Aryan's character serves as a catalyst for Rohan's growth, providing encouragement and support throughout the narrative.

**Themes**

The script explores several themes, including:

1. **Ambition vs Reality**: Rohan's struggles to balance his ambitions with the harsh realit